# 소행성 관측 계획 수립하기

* 이 노트북을 구글 코랩에서 실행하고자 한다면 [파일] - [드라이브에 사본 저장]을 하여 본인의 소유로 만든 후에 코드를 실행하거나 수정할 수 있습니다.

* 이 파일은 실제 수업에 사용하므로 필요에 따라 예고 없이 변경될 수 있습니다.

* If you have any questions or comments on this document, please email me(Kiehyun.Park@gmail.com).

* 이 파일(문서)는 공교육 현장에서 수업시간에 자유롭게 사용할 수 있으나, 다른 목적으로 사용할 시에는 사전에 연락을 주셔서 상의해 주시기 바랍니다.

이 자료는 What's observable로 부터 특정한 날 관측할 수 있는 소행성의 목록을 만드는 과정을 설명합니다.

## 필요한 환경

이 프로젝트를 위해서는 아래의 모듈이 필요합니다.

> numpy, pandas, matplotlib, astroquery, requests, astropy, astropy.units, certifi, version_information


### 구글 코랩에 한글 폰트 설치

matplotlib에서 한글을 사용하기 위해서는 한글 폰트가 필요하다. 구글 코랩에서 현재의 Jupyter notebook을 실행한다면 아래 코드를 실행 한 후 런타임 다시 시작을 해 줘야 한글을 사용할 수 있을 것이다.

In [1]:
# !sudo apt-get install -y fonts-nanum
# !sudo fc-cache -fv
# !rm ~/.cache/matplotlib -rf

'sudo'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
'sudo'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
'rm'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


### 런타임 다시 시작

위의 셀을 실행한 다음 반드시 다음 과정을 잊지 말자.

* [메뉴]-[런타임]-[런터임 다시 시작]

* [메뉴]-[런타임]-[이전 셀 실행]을 해주어야 한다.

### 한글 폰트 사용

위에서 한글 폰트를 설치하고, 런타임 다시시작을 했다면 구글 코랩에서 폰트 경로를 설정하여 한글 사용이 가능해 진다.

In [2]:
#visualization
import matplotlib as mpl
import matplotlib.pylab as plb
import matplotlib.pyplot as plt

# 브라우저에서 바로 그려지도록
%matplotlib inline

# 그래프에 retina display 적용
%config InlineBackend.figure_format = 'retina'

# Colab 의 한글 폰트 설정
plt.rc('font', family='NanumBarunGothic')

# 유니코드에서  음수 부호설정
mpl.rc('axes', unicode_minus=False)

### 모듈 설치 및 버전 확인

아래 셀을 실행하면 이 노트북을 실행하는데 필요한 모듈을 설치하고 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [3]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, astroquery, requests, astropy, astropy.units, certifi, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"**** {pkg} module is now installed.")
    else:
        print(f"******** {pkg} module is already installed.")
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

******** numpy module is already installed.
******** pandas module is already installed.
******** matplotlib module is already installed.
******** astroquery module is already installed.
******** requests module is already installed.
******** astropy module is already installed.
******** astropy.units module is already installed.
******** certifi module is already installed.
******** version_information module is already installed.
This notebook was generated at 2023-11-13 20:12:16 (대한민국 표준시 = GMT+0900) 
0 Python     3.11.5 64bit [MSC v.1916 64 bit (AMD64)]
1 IPython    8.15.0
2 OS         Windows 10 10.0.22631 SP0
3 numpy      1.26.0
4 pandas     2.0.3
5 matplotlib 3.7.2
6 astroquery 0.4.6
7 requests   2.31.0
8 astropy    5.1
9 astropy.units The 'astropy.units' distribution was not found and is required by the application
10 certifi    2023.07.22
11 version_information 1.0.4


## 날짜와 시간 다루기

### datetime.datetime

Python 내장 모듈인 datetime을 사용하면 날짜와 시각 객체를 다룰 수 있습니다.

#### 표준시와 세계시

In [4]:
from datetime import datetime, timedelta, timezone

dt_kst_now = datetime.now()
print("현재 시각(KST=GMT+9) :", dt_kst_now)

timezone_utc = timezone(timedelta(hours=-9))
dt_utc_now = dt_kst_now.astimezone(timezone_utc)
print("현재 시각(UTC) :", dt_utc_now)

현재 시각(KST=GMT+9) : 2023-11-13 20:12:17.755207
현재 시각(UTC) : 2023-11-13 02:12:17.755207-09:00


#### fits header의 시간 형식

fits header의 시간 형식은 '2023-09-11T20:19:18'의 형태를 띠고 있는데 datetime.strptime을 이용하면 날짜시간 형식을 바로 datetime 객체로 만들 수 있습니다.

이를 세계시로 바꾸려면 timedelta를 이용할 수 있습니다.

In [5]:
start_dt_kst_str = '2023-09-11T20:19:18'
end_dt_kst_str = '2023-12-31T00:00:00'

start_dt_kst = datetime.strptime(start_dt_kst_str, '%Y-%m-%dT%H:%M:%S')
print(start_dt_kst)
print(type(start_dt_kst))

start_dt_utc = start_dt_kst + timedelta(hours=-9)
print(start_dt_utc)
print(type(start_dt_utc))

2023-09-11 20:19:18
<class 'datetime.datetime'>
2023-09-11 11:19:18
<class 'datetime.datetime'>


#### string 형태로 변환하기

datetime 객체를 string으로 바꿀때는 strftime 메쏘들ㄹ 이용하면 됩니다.

In [6]:

start_dt_utc_str = start_dt_utc.strftime('%Y-%m-%dT%H:%M:%S')
print("start_dt_utc_str :", start_dt_utc_str)
print("type(start_dt_utc_str) :", type(start_dt_utc_str))


start_dt_utc_str : 2023-09-11T11:19:18
type(start_dt_utc_str) : <class 'str'>


#### datetime 이용하여 list 생성

아래와 같이 하면, 시작 시각과 끝나는 시각, 그리고 시간 간격을 정해 주고, Python의 datatime 객체를 이용하여 일정 시간 간격의 목록(list)을 만을 수 있다.

In [7]:
from datetime import datetime, timedelta
start_time_text = "2017-01-01"
end_time_text = "2017-06-01"
dt = 1
def datetime_range(start_dt, end_dt, delta):
    '''
        Parameters
        ----------
        start_dt : datetime.datetime
        end_dt : datetime.datetime
        delta : int
    '''
    current = start_dt
    while current < end_dt:
        yield current
        current += delta

dts = [dt.strftime('%Y-%m-%d') for dt in
            datetime_range(datetime.fromisoformat(start_time_text), datetime.fromisoformat(end_time_text),
            timedelta(days=dt))]

print("len(dts):", len(dts))
print("dts", dts)

len(dts): 151
dts ['2017-01-01', '2017-01-02', '2017-01-03', '2017-01-04', '2017-01-05', '2017-01-06', '2017-01-07', '2017-01-08', '2017-01-09', '2017-01-10', '2017-01-11', '2017-01-12', '2017-01-13', '2017-01-14', '2017-01-15', '2017-01-16', '2017-01-17', '2017-01-18', '2017-01-19', '2017-01-20', '2017-01-21', '2017-01-22', '2017-01-23', '2017-01-24', '2017-01-25', '2017-01-26', '2017-01-27', '2017-01-28', '2017-01-29', '2017-01-30', '2017-01-31', '2017-02-01', '2017-02-02', '2017-02-03', '2017-02-04', '2017-02-05', '2017-02-06', '2017-02-07', '2017-02-08', '2017-02-09', '2017-02-10', '2017-02-11', '2017-02-12', '2017-02-13', '2017-02-14', '2017-02-15', '2017-02-16', '2017-02-17', '2017-02-18', '2017-02-19', '2017-02-20', '2017-02-21', '2017-02-22', '2017-02-23', '2017-02-24', '2017-02-25', '2017-02-26', '2017-02-27', '2017-02-28', '2017-03-01', '2017-03-02', '2017-03-03', '2017-03-04', '2017-03-05', '2017-03-06', '2017-03-07', '2017-03-08', '2017-03-09', '2017-03-10', '2017-03-11', '

### astropy.time

표준시나 세계시 정도는 datetime을 사용해도 무방하지만 천문학에서 사용하는 시간에는 조금 부족합니다. 천문학에서 사용하는 시간계는 datetime 모듈보다는 astropy.time 모듈을 사용하기를 권장합니다.

#### 세계시

In [8]:
from astropy.time import Time

t_kst_now = Time.now()
print("현재 시각(KST=GMT+9) :", t_kst_now)

t_ut_now = Time(datetime.utcnow(), scale='utc', location=('127d', '37d'))
print("현재 시각(UTC) :", t_ut_now)

print("type(t_ut_now) ;", type(t_ut_now))

현재 시각(KST=GMT+9) : 2023-11-13 11:12:17.826016


현재 시각(UTC) : 2023-11-13 11:12:17.828049
type(t_ut_now) ; <class 'astropy.time.core.Time'>


#### dir함수로 메쏘드 알아보기


In [9]:
print("dir(t_ut_now) ;", dir(t_ut_now))

dir(t_ut_now) ; ['FORMATS', 'SCALES', 'T', '_APPLICABLE_FUNCTIONS', '_METHOD_FUNCTIONS', '__abstractmethods__', '__add__', '__array_function__', '__array_priority__', '__bool__', '__class__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__radd__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__sub__', '__subclasshook__', '__weakref__', '_abc_impl', '_advanced_index', '_apply', '_astropy_column_attrs', '_call_erfa', '_format', '_get_delta_tdb_tt', '_get_delta_ut1_utc', '_get_time_fmt', '_init_from_vals', '_make_value_equivalent', '_match_shape', '_set_delta_tdb_tt', '_set_delta_ut1_utc', '_set_scale', '_shaped_like_input', '_sid_time_or_earth_rot_ang', '_

#### Julian day 구하기

In [10]:
print("t_ut_now.to_value(format='jd') ;", t_ut_now.to_value(format='jd'))

t_ut_now.to_value(format='jd') ; 2460261.9668730097


#### 항성시

In [11]:
print("t_ut_now.sidereal_time('mean') :", t_ut_now.sidereal_time('mean'))

t_ut_now.sidereal_time('mean') : 23h09m33.69446851s


#### astropy.time 이용하여 list 생성

일정 시간 간격으로 시뮬레이션 하려면 시간 간격의 리스트를 미리 만들어 두면 편리할 것입니다. 다음은 Python의 astropy.time을 이용하여 일정 시간 간격의 목록(list)을 만드는 방법입니다.

In [12]:
import numpy as np
from astropy.time import Time, TimeDelta

start_time_text = "2023-10-06"
end_time_text = "2023-10-15"
dt = 1

def astropyTime_range(start_t, end_t, delta):
    '''
        Parameters
        ----------
        start_t : datetime.datetime
        end_t : datetime.datetime
        delta : int
    '''
    current = start_t
    while current < end_t:
        yield current
        current += delta

dts = [dt for dt in
        astropyTime_range(Time(start_time_text), Time(end_time_text),
        TimeDelta(dt, format='jd'))]

print(len(dts))
print(dts)

9
[<Time object: scale='utc' format='iso' value=2023-10-06 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-10-07 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-10-08 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-10-09 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-10-10 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-10-11 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-10-12 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-10-13 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-10-14 00:00:00.000>]


## What's observable

[What's observable](https://ssd.jpl.nasa.gov/tools/sbwobs.html) 에서는 관측소 코드와 관측 조건들을 설정하여 소행성과 혜성의 관측 대상의 정보를 얻을 수 있습니다.


### Horizons API

[Horizons API](https://ssd-api.jpl.nasa.gov/doc/horizons.html)를 제공하고 있어 requests로 받아아올 url을 만들수 있습니다.



#### 검색 조건 설정

먼저 조건들을 변수로 설정해 보겠습니다.

In [13]:
#############################################
# variables
mpc_code='P64' # Observer Location: GSHS Observatory, Suwon [code: P64]

#관측 시각 설정 (현재 시각)
t_ut_now = Time(datetime.utcnow(), scale='utc', location=('127d', '37d'))
print("type(obs_dt_utc) :", type(t_ut_now))
print("현재 시각(UTC) :", t_ut_now)

#관측 시각 설정 (임의의 시각)
# t_ut_now = Time(datetime(2023, 10, 2, 10, 0, 0), scale='utc', location=('127d', '37d'))
# print("type(obs_dt_utc) :", type(t_ut_now))
# print("현재 시각(UTC) :", t_ut_now)

obs_datetime = t_ut_now.strftime('%Y-%m-%d %H:%M:%S')
print("obs_datetime :", obs_datetime)
obs_date = t_ut_now.strftime('%Y%m%d')
print("obs_date :", obs_date)

elev_min=60  #
time_min=15
vmag_min=6
vmag_max=13
list_num=100
sort='trans'

type(obs_dt_utc) : <class 'astropy.time.core.Time'>
현재 시각(UTC) : 2023-11-13 11:12:40.310594
obs_datetime : 2023-11-13 11:12:40
obs_date : 20231113


#### url 만들기

In [14]:
# Define API URL and SPK filename:
url = 'https://ssd-api.jpl.nasa.gov/sbwobs.api'
spk_filename = 'spk_file.bsp'

# Get the requested SPK-ID from the command-line:
if (len(sys.argv)) == 1:
  print("please specify SPK-ID on the command-line")
  sys.exit(2)
spkid = sys.argv[1]

# Build the appropriate URL for this API request:
# IMPORTANT: You must encode the "=" as "%3D" and the ";" as "%3B" in the
#            Horizons COMMAND parameter specification.
url += f"?sb-kind=a&mpc-code={mpc_code}&obs-time={str(obs_datetime)}&elev-min={elev_min}&time-min={time_min}"
url += f"&vmag-max={vmag_max}&vmag-min={vmag_min}&optical=true&fmt-ra-dec=true&mag-required=true&output-sort={sort}&maxoutput={list_num}"

print("url :", url)

url : https://ssd-api.jpl.nasa.gov/sbwobs.api?sb-kind=a&mpc-code=P64&obs-time=2023-11-13 11:12:40&elev-min=60&time-min=15&vmag-max=13&vmag-min=6&optical=true&fmt-ra-dec=true&mag-required=true&output-sort=trans&maxoutput=100


### reauests.get

api에서 제공하는 내용을 json 형태로 받아 보겠습니다.

In [15]:
import json
import requests
# Submit the API request and decode the JSON-response:
response = requests.get(url)
try:
  data = json.loads(response.text)
  print(data)
except ValueError:
  print("Unable to decode JSON results")

{'signature': {'source': 'NASA/JPL Small-Body Observability API', 'version': '1.0'}, 'obs_constraints': {'optical': 'true', 'vmag-min': '6', 'elev-min': '60', 'vmag-max': '13', 'output-sort': 'trans', 'obs-time': '2023-11-13 11:12:40', 'mag-required': 'true', 'maxoutput': '100', 'time-min': '15', 'mpc-code': 'P64', 'fmt-ra-dec': 'true'}, 'sb_constraints': {'sb-kind': 'a'}, 'location': 'GSHS Observatory, Suwon', 'obs_night': {'sun_set_az': '248.4', 'sun_set': '2023-11-12 08:23', 'end_civil': '2023-11-12 08:51', 'end_nautical': '2023-11-12 09:22', 'end_astronomical': '2023-11-12 09:53', 'begin_astronomical': '2023-11-12 20:36', 'begin_nautical': '2023-11-12 21:07', 'begin_civil': '2023-11-12 21:39', 'sun_rise_az': '111.8', 'sun_rise': '2023-11-12 22:06', 'transit_phase': '0.00', 'transit': '1984-10-26 23:58 ', 'moon_set_phase': '0.01', 'moon_set': '2023-11-12 07:35 ', 'moon_rise_phase': '0.00', 'moon_rise': '2023-11-12 21:46 ', 'begin_dark': '2023-11-12 09:53', 'mid_dark': '2023-11-12 15

안에 들어 있는 데이터를 확인해 보곘습니다.

In [16]:
print("type(data) :", type(data))
print("data.keys :", data.keys)

type(data) : <class 'dict'>
data.keys : <built-in method keys of dict object at 0x00000173BEC97D80>


 아마도 data['fields']에 있는 것이 필요한 자료인것 같습니다.

In [17]:
print(data['obs_constraints'])
print(data['sb_constraints'])
print(data['fields'])
print(data['data'])
print(type(data['fields']))
print(len(data['fields']))
print(len(data['data'][0]))


{'optical': 'true', 'vmag-min': '6', 'elev-min': '60', 'vmag-max': '13', 'output-sort': 'trans', 'obs-time': '2023-11-13 11:12:40', 'mag-required': 'true', 'maxoutput': '100', 'time-min': '15', 'mpc-code': 'P64', 'fmt-ra-dec': 'true'}
{'sb-kind': 'a'}
['Designation', 'Full name', 'Rise time', 'Transit time', 'Set time', 'Max. time observable', 'R.A.', 'Dec.', 'Vmag', 'Helio. range (au)', 'Topo.range (au)', 'Object-Observer-Sun (deg)', 'Object-Observer-Moon (deg)', 'Galactic latitude (deg)']
[['602', '602 Marianna (A906 DJ)', '10:09', '11:27', '12:46', '02:36', '23:20:57', '+12 58\'53"', '12.6', '2.31', '1.59', '124.75', '134.0', '-44.25'], ['678', '678 Fredegundis (A909 BQ)', '11:04', '12:01', '12:58', '01:54', '23:55:00', '+10 14\'50"', '12.4', '2.02', '1.22', '132.21', '141.5', '-50.22'], ['120', '120 Lachesis (A872 GB)', '11:41', '13:03', '14:24', '02:43', '00:56:50', '+13 30\'38"', '13.0', '3.28', '2.40', '147.67', '157.0', '-49.48'], ['639', '639 Latona (A907 OA)', '11:24', '13:17

### dataframe 만들기

아무래도 가장 다루기 쉬운 데이타 형태인 dataframe으로 만들어 보겠습니다.

In [18]:
import pandas as pd

df = pd.DataFrame.from_dict(data['data'])
df = df.set_axis((data['fields']), axis=1)
df['OBSdate(UT)'] = obs_date
df

,Designation,Full name,Rise time,Transit time,Set time,Max. time observable,R.A.,Dec.,Vmag,Helio. range (au),Topo.range (au),Object-Observer-Sun (deg),Object-Observer-Moon (deg),Galactic latitude (deg),OBSdate(UT)
0,602,602 Marianna (A906 DJ),10:09,11:27,12:46,02:36,23:20:57,"+12 58'53""",12.6,2.31,1.59,124.75,134.0,-44.25,20231113
1,678,678 Fredegundis (A909 BQ),11:04,12:01,12:58,01:54,23:55:00,"+10 14'50""",12.4,2.02,1.22,132.21,141.5,-50.22,20231113
2,120,120 Lachesis (A872 GB),11:41,13:03,14:24,02:43,00:56:50,"+13 30'38""",13.0,3.28,2.40,147.67,157.0,-49.48,20231113
3,639,639 Latona (A907 OA),11:24,13:17,15:11,03:47,01:11:31,"+20 21'26""",12.6,2.76,1.85,151.88,160.8,-42.43,20231113
4,123,123 Brunhild (A872 OB),11:36,13:28,15:20,03:44,01:22:14,"+19 55'02""",12.5,2.42,1.49,154.40,163.3,-42.54,20231113
5,135,135 Hertha (A874 DA),12:17,13:33,14:49,02:31,01:27:15,"+12 34'47""",11.3,2.18,1.25,154.67,164.0,-49.54,20231113
6,122,122 Gerda (A872 OA),13:02,13:45,14:28,01:26,01:39:40,"+08 56'25""",12.6,3.25,2.32,156.33,165.2,-52.29,20231113
7,127,127 Johanna (A872 VB),12:42,14:10,15:39,02:56,02:04:45,"+14 38'08""",12.5,2.73,1.76,164.01,173.3,-44.83,20231113
8,509,509 Iolanda (A903 HD),12:58,14:10,15:23,02:24,02:04:33,"+12 05'15""",12.4,2.79,1.83,163.19,172.1,-47.19,20231113
9,595,595 Polyxena (A906 FL),12:19,14:17,16:14,03:55,02:10:56,"+21 30'34""",12.7,3.23,2.26,165.56,173.2,-37.92,20231113


### 관측 계획 저장 폴더 생성

FITS 파일을 저장할 폴더를 "OBSplan_asteroid" 이라는 이름으로 생성해보겠습니다.

* 만약 리눅스 시스템 이라면 shell 명령어로 가능한데, "!"를 붙이면 shell 명령어를 실행할 수 있습니다.
> !mkdir OBSplan_asteroid

OS의 영향을 받지 않기 위하여 pathlib을 사용하여 다음과 같이 폴더를 생성합니다.

In [19]:
import os
from pathlib import Path
BASEPATH = Path("./")
save_dir_name = "OBSplan_asteroid"
print(f"BASEPATH: {BASEPATH}")

if not (BASEPATH/save_dir_name).exists():
    os.mkdir(str(BASEPATH/save_dir_name))
    print (f"{str(BASEPATH/save_dir_name)} is created...")
else :
    print (f"{str(BASEPATH/save_dir_name)} is already exist...")

BASEPATH: .
OBSplan_asteroid is created...


### dataframe 저장

In [20]:
df.to_csv(str(BASEPATH/save_dir_name/ f'asteroid_{obs_date}.csv'))

### (과제)

12월 1달 동안의 NEA 목록을 csv  파일로 저장하는 코드를 완성하세요.



In [21]:
#(과제) 아래 코딩을 완성하여 제출하시오.
#############################################
# variables
mpc_code='P64' # Observer Location: GSHS Observatory, Suwon [code: P64]

elev_min=60 #
time_min=15
vmag_min=6
vmag_max=13
list_num=50
sort='trans'
#############################################

start_dt = datetime.strptime("2023-11-13", '%Y-%m-%d')
end_dt = datetime.strptime("2023-11-14", '%Y-%m-%d')
dt = 1

dts = [dt for dt in
            datetime_range(start_dt, end_dt,
            timedelta(days=dt))]

print("len(dts):", len(dts))
print("dts", dts)

df_all = pd.DataFrame()

for dt in dts :
    obs_datetime = dt.strftime('%Y-%m-%d %H:%M:%S')
    print("obs_datetime :", obs_datetime)
    obs_date = dt.strftime('%Y%m%d')
    print("obs_date :", obs_date)

    # Define API URL and SPK filename:
    url = 'https://ssd-api.jpl.nasa.gov/sbwobs.api'
    spk_filename = 'spk_file.bsp'

    # Get the requested SPK-ID from the command-line:
    if (len(sys.argv)) == 1:
        print("please specify SPK-ID on the command-line")
        sys.exit(2)
    spkid = sys.argv[1]

    # Build the appropriate URL for this API request:
    # IMPORTANT: You must encode the "=" as "%3D" and the ";" as "%3B" in the
    #            Horizons COMMAND parameter specification.
    url += f"?sb-kind=a&mpc-code={mpc_code}&obs-time={str(obs_datetime)}&elev-min={elev_min}&time-min={time_min}"
    url += f"&vmag-max={vmag_max}&vmag-min={vmag_min}&optical=true&fmt-ra-dec=true&mag-required=true&output-sort={sort}&maxoutput={list_num}"

    print("url :", url)

    # Submit the API request and decode the JSON-response:
    response = requests.get(url)
    try:
        data = json.loads(response.text)
        print(data)
    except ValueError:
        print("Unable to decode JSON results")

    df = pd.DataFrame.from_dict(data['data'])
    df = df.set_axis((data['fields']), axis=1)
    df['OBSdate(UT)'] = obs_date
    #print(df)

    df.to_csv(str(BASEPATH/save_dir_name/f"asteroid_{dt.strftime('%Y%m%d')}.csv"))

    df_all = pd.concat([df_all, df], axis = 0)

df_all.reset_index(inplace=True)
print("df_all :", df_all)

df_all.to_csv(str(BASEPATH/save_dir_name/f"asteroid_{dts[0].strftime('%Y%m%d')}-{dts[-1].strftime('%Y%m%d')}.csv"))

len(dts): 1
dts [datetime.datetime(2023, 11, 13, 0, 0)]
obs_datetime : 2023-11-13 00:00:00
obs_date : 20231113
url : https://ssd-api.jpl.nasa.gov/sbwobs.api?sb-kind=a&mpc-code=P64&obs-time=2023-11-13 00:00:00&elev-min=60&time-min=15&vmag-max=13&vmag-min=6&optical=true&fmt-ra-dec=true&mag-required=true&output-sort=trans&maxoutput=50


{'signature': {'source': 'NASA/JPL Small-Body Observability API', 'version': '1.0'}, 'obs_constraints': {'vmag-min': '6', 'elev-min': '60', 'vmag-max': '13', 'optical': 'true', 'mag-required': 'true', 'maxoutput': '50', 'obs-time': '2023-11-13 00:00:00', 'output-sort': 'trans', 'fmt-ra-dec': 'true', 'mpc-code': 'P64', 'time-min': '15'}, 'sb_constraints': {'sb-kind': 'a'}, 'location': 'GSHS Observatory, Suwon', 'obs_night': {'sun_set_az': '248.4', 'sun_set': '2023-11-12 08:23', 'end_civil': '2023-11-12 08:51', 'end_nautical': '2023-11-12 09:22', 'end_astronomical': '2023-11-12 09:53', 'begin_astronomical': '2023-11-12 20:36', 'begin_nautical': '2023-11-12 21:07', 'begin_civil': '2023-11-12 21:39', 'sun_rise_az': '111.8', 'sun_rise': '2023-11-12 22:06', 'transit_phase': '0.00', 'transit': '1984-10-26 23:58 ', 'moon_set_phase': '0.01', 'moon_set': '2023-11-12 07:35 ', 'moon_rise_phase': '0.00', 'moon_rise': '2023-11-12 21:46 ', 'begin_dark': '2023-11-12 09:53', 'mid_dark': '2023-11-12 15:

## 소행성 좌표 얻기

### 소행성 id 잘라내기

앞서 얻은 dataframe로 부터 소행성의 이름을 알아보자.

In [22]:
#print("df :", df)
#print("df['Full name'] :", df['Full name'])
print("df['Full name'][0] :", df['Full name'][0])

#import re
#asteroid_names = re.findall('\(([^)]+)', df['Full name'][0])   #extracts string in bracket()
#print (asteroid_names[0])

asteroid_id = df['Full name'][0].split(" ")[0]   #extracts string in bracket()
print("asteroid_id :", asteroid_id)
obs_date = df['OBSdate(UT)'][0]

df['Full name'][0] : 602 Marianna (A906 DJ)
asteroid_id : 602


### astroquery.mpc

[astroquery.mpc](https://astroquery.readthedocs.io/en/latest/mpc/mpc.html) 를 이용하여 시간에 따른 소행성의 좌표를 얻어보자.

In [23]:
from astroquery.mpc import MPC
eph = MPC.get_ephemeris(asteroid_id, step='1h', number=30 * 24)
print("type(eph) :", type(eph))
print("eph :", eph)
df_eph = eph.to_pandas()
df_eph

type(eph) : <class 'astropy.table.table.Table'>
eph :           Date                  RA         ... Uncertainty 3sig Unc. P.A.
                               deg         ...      arcsec         deg   
----------------------- ------------------ ... ---------------- ---------
2023-11-13 11:00:00.000  349.9858333333333 ...                0      48.9
2023-11-13 12:00:00.000  349.9879166666666 ...                0      48.9
2023-11-13 13:00:00.000 349.99041666666665 ...                0      48.9
2023-11-13 14:00:00.000 349.99291666666664 ...                0      48.9
2023-11-13 15:00:00.000  349.9954166666666 ...                0      48.9
2023-11-13 16:00:00.000 349.99791666666664 ...                0      48.9
2023-11-13 17:00:00.000  350.0004166666666 ...                0      48.9
2023-11-13 18:00:00.000 350.00291666666664 ...                0      48.9
2023-11-13 19:00:00.000 350.00541666666663 ...                0      48.9
2023-11-13 20:00:00.000  350.0079166666666 ...            

,Date,RA,Dec,Delta,r,Elongation,Phase,V,Proper motion,Direction,Uncertainty 3sig,Unc. P.A.
0,2023-11-13 11:00:00,349.985833,12.858333,1.603,2.307,124.0,20.8,12.6,8.69,79.3,0,48.9
1,2023-11-13 12:00:00,349.987917,12.858889,1.603,2.307,124.0,20.8,12.6,8.74,79.3,0,48.9
2,2023-11-13 13:00:00,349.990417,12.859167,1.603,2.307,123.9,20.9,12.6,8.78,79.3,0,48.9
3,2023-11-13 14:00:00,349.992917,12.859722,1.604,2.307,123.9,20.9,12.6,8.83,79.3,0,48.9
4,2023-11-13 15:00:00,349.995417,12.860278,1.604,2.307,123.9,20.9,12.6,8.88,79.3,0,48.9
...,...,...,...,...,...,...,...,...,...,...,...,...
715,2023-12-13 06:00:00,354.715417,14.033611,1.932,2.314,99.9,24.8,13.1,37.27,73.9,0,49.8
716,2023-12-13 07:00:00,354.725417,14.036389,1.932,2.314,99.9,24.8,13.1,37.31,73.9,0,49.8
717,2023-12-13 08:00:00,354.735833,14.039444,1.933,2.314,99.8,24.8,13.1,37.34,73.9,0,49.8
718,2023-12-13 09:00:00,354.746250,14.042222,1.933,2.314,99.8,24.8,13.1,37.37,73.9,0,49.8


### 최대 움직임 속도 알아보기

In [24]:
print(f"Proper motion: {eph['Proper motion'].quantity.to('arcmin/h').max():.03f}")

Proper motion: 0.623 arcmin / h


### 파일 저장하기

In [25]:
eph = MPC.get_ephemeris(asteroid_id, location='P64',
                        ra_format={'sep': ':', 'unit': 'hourangle', 'precision': 1},
                        dec_format={'sep': ':', 'precision': 0},
                        step='1h')
print("type(eph): ", type(eph))
#print("eph: ", eph)
df_eph = eph.to_pandas()
df_eph['Asteroid ID'] = asteroid_id
df_eph['Asteroid name'] = df['Full name'][0]
df_eph.to_csv(str(BASEPATH/save_dir_name/ f'asteroid_{asteroid_id}.csv'))
df_eph

type(eph):  <class 'astropy.table.table.Table'>


,Date,RA,Dec,Delta,r,Elongation,Phase,V,Proper motion,Direction,Azimuth,Altitude,Sun altitude,Moon phase,Moon distance,Moon altitude,Uncertainty 3sig,Unc. P.A.,Asteroid ID,Asteroid name
0,2023-11-13 11:00:00,23:19:56.6,12:51:28,1.602,2.307,124.0,20.8,12.6,7.58,77.5,167,65,-31,0.00,124,-33,0,48.9,602,602 Marianna (A906 DJ)
1,2023-11-13 12:00:00,23:19:57.1,12:51:29,1.603,2.307,124.0,20.8,12.6,7.62,78.0,201,64,-43,0.00,124,-45,0,48.9,602,602 Marianna (A906 DJ)
2,2023-11-13 13:00:00,23:19:57.6,12:51:31,1.603,2.307,123.9,20.9,12.6,7.74,78.6,228,57,-55,0.00,123,-56,0,48.9,602,602 Marianna (A906 DJ)
3,2023-11-13 14:00:00,23:19:58.1,12:51:33,1.604,2.307,123.9,20.9,12.6,7.93,79.2,246,47,-65,0.00,122,-66,0,48.9,602,602 Marianna (A906 DJ)
4,2023-11-13 15:00:00,23:19:58.7,12:51:34,1.604,2.307,123.9,20.9,12.6,8.18,79.8,258,36,-71,0.00,122,-73,0,48.9,602,602 Marianna (A906 DJ)
5,2023-11-13 16:00:00,23:19:59.2,12:51:35,1.605,2.307,123.8,20.9,12.6,8.48,80.3,268,24,-68,0.00,121,-72,0,48.9,602,602 Marianna (A906 DJ)
6,2023-11-13 17:00:00,23:19:59.8,12:51:37,1.605,2.307,123.8,20.9,12.6,8.82,80.7,277,12,-60,0.00,121,-64,0,48.9,602,602 Marianna (A906 DJ)
7,2023-11-13 18:00:00,23:20:00.4,12:51:38,1.605,2.307,123.7,20.9,12.6,9.16,81.0,286,0,-49,0.00,120,-54,0,48.9,602,602 Marianna (A906 DJ)
8,2023-11-13 19:00:00,23:20:01.1,12:51:40,1.606,2.307,123.7,20.9,12.6,9.49,81.1,296,-11,-37,0.00,120,-42,0,48.9,602,602 Marianna (A906 DJ)
9,2023-11-13 20:00:00,23:20:01.7,12:51:41,1.606,2.307,123.7,20.9,12.6,9.80,81.1,306,-21,-25,0.00,119,-31,0,48.9,602,602 Marianna (A906 DJ)


### (과제)

df에 있는 소행성 목록들의 eph를 csv 파일로 저장해 보자.

In [26]:
from pathlib import Path
from astroquery.mpc import MPC
from pprint import pprint
import re

for i, row  in df.iterrows():
    print("row['Full name'] :", row["Full name"])

    #asteroid_names = re.findall('\(([^)]+)', df['Full name'][0])   #extracts string in bracket()
    #print (asteroid_names[0])
    asteroid_id = df['Full name'][i].split(" ")[0]   #extracts string
    print(asteroid_id)

    eph = MPC.get_ephemeris(asteroid_id, step='1h', number=30 * 24)
    print(f"Proper motion: {eph['Proper motion'].quantity.to('arcmin/h').max():.03f}")
    eph = MPC.get_ephemeris(asteroid_id, location='P64',
                        ra_format={'sep': ':', 'unit': 'hourangle', 'precision': 1},
                        dec_format={'sep': ':', 'precision': 0},
                        step='1h')
    print("type(eph): ", type(eph))
    #print("eph: ", eph)

    df_eph = eph.to_pandas()
    df_eph['Asteroid ID'] = asteroid_id
    df_eph['Asteroid name'] = df['Full name'][i]
    df_eph.to_csv(str(BASEPATH/save_dir_name/ f"eph_asteroid_{asteroid_id}.csv"))

row['Full name'] : 602 Marianna (A906 DJ)
602
Proper motion: 0.623 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 678 Fredegundis (A909 BQ)
678
Proper motion: 0.681 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 120 Lachesis (A872 GB)
120
Proper motion: 0.323 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 639 Latona (A907 OA)
639
Proper motion: 0.394 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 123 Brunhild (A872 OB)
123
Proper motion: 0.435 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 135 Hertha (A874 DA)
135
Proper motion: 0.379 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 122 Gerda (A872 OA)
122
Proper motion: 0.389 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 127 Johanna (A872 VB)
127
Proper motion: 0.527 arcmin / h
type(eph):  <class 'astropy.table.table.Table

저장

In [27]:
eph = MPC.get_ephemeris('A894 VC', start='1996-03-01', step='1h', number=30 * 24)
print(eph['Proper motion'].quantity.to('deg/h').max())
eph = MPC.get_ephemeris('A894 VC', location='P64',
                        ra_format={'sep': ':', 'unit': 'hourangle', 'precision': 1},
                        dec_format={'sep': ':', 'precision': 0},
                        step='1h')
print(eph)

0.023877777777777776 deg / h
          Date              RA       Dec    ... Uncertainty 3sig Unc. P.A.
                        hourangle    deg    ...      arcsec         deg   
----------------------- ---------- -------- ... ---------------- ---------
2023-11-13 11:00:00.000 20:46:22.9 -5:18:33 ...                0      77.0
2023-11-13 12:00:00.000 20:46:28.0 -5:18:33 ...                0      76.9
2023-11-13 13:00:00.000 20:46:33.0 -5:18:33 ...                0      76.9
2023-11-13 14:00:00.000 20:46:38.1 -5:18:33 ...                0      77.0
2023-11-13 15:00:00.000 20:46:43.1 -5:18:33 ...                0      76.9
2023-11-13 16:00:00.000 20:46:48.2 -5:18:33 ...                0      77.0
2023-11-13 17:00:00.000 20:46:53.3 -5:18:33 ...                0      76.9
2023-11-13 18:00:00.000 20:46:58.5 -5:18:33 ...                0      77.0
2023-11-13 19:00:00.000 20:47:03.6 -5:18:33 ...                0      77.0
2023-11-13 20:00:00.000 20:47:08.8 -5:18:33 ...                0      7